# RewindKV — Sonda nos estados internos (teste de viabilidade)

O estado interno do modelo no instante da detecção separa **erro real** de **falso alarme / lado errado**? Protocolo e critério pré-registrados em `docs/resultados-experimentos.md` (seção *Sonda nos estados internos*).

**Antes de rodar:**
1. *Ambiente de execução → Alterar o tipo de ambiente → **GPU T4***.
2. Na barra lateral, em 🔑 *Secrets*, o `GITHUB_TOKEN` precisa estar com o acesso ativado para este notebook.
3. Tenha à mão o arquivo `dados_sonda.zip` (está na sua pasta Downloads).

**Para rodar:** *Ambiente de execução → Executar tudo*. A célula 3 pede o upload do `dados_sonda.zip`. Leva uns 15 minutos; no fim, baixa um `resultados_sonda.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código do projeto
Clona a tag `sonda-v1`: o código fica congelado no estado do pré-registro.

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "GioNicolas/REWIND-KV"
TAG = "sonda-v1"
DEST = "/content/REWIND-KV"

if not os.path.isdir(DEST):
    token = userdata.get("GITHUB_TOKEN")
    r = subprocess.run(["git", "clone", "-q", "--branch", TAG, f"https://{token}@github.com/{REPO}.git", DEST],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin", f"https://github.com/{REPO}.git"], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd {DEST}
!pip install -q -e ".[dev,probe]"
!python -m pytest -q

## 2. Dados das validações
Escolha o arquivo `dados_sonda.zip` quando o botão de upload aparecer.

In [ ]:
import zipfile, glob
from google.colab import files

DATA = "/content/dados_sonda"
if not os.path.isdir(DATA):
    uploaded = files.upload()
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(DATA)
print({d: len(glob.glob(f"{DATA}/{d}/*.jsonl")) for d in ("nivel4", "nivel5")}, "(esperado: 2 e 2)")

## 3. Estados internos de cada corte (GPU)
Uma passada do modelo por corte: 69 do nível 4 e 84 do nível 5.

In [ ]:
OUT = "/content/sonda_out"
os.makedirs(OUT, exist_ok=True)
for level, folder, labels in [(4, "nivel4", "validacao-v02"), (5, "nivel5", "validacao-v03")]:
    !python -m rewind_kv.probe extract --runs {DATA}/{folder} --labels docs/dados/rotulos-cortes-{labels}.json \
        --problems {DATA}/math500_test.jsonl --level {level} --out {OUT}/feats_n{level}.npz > {OUT}/log_n{level}.txt 2>&1
    !tail -n 3 {OUT}/log_n{level}.txt

## 4. Avaliação pré-registrada
Escolhe a configuração só no nível 4 (validação cruzada) e testa **uma vez** no nível 5. Critério: precisão ≥ 75% **e** recall ≥ 70% no nível 5.

In [ ]:
!python -m rewind_kv.probe evaluate --train {OUT}/feats_n4.npz --test {OUT}/feats_n5.npz | tee {OUT}/avaliacao.txt

## 5. Baixar os resultados

In [ ]:
!cd /content && zip -q -r resultados_sonda.zip sonda_out
files.download("/content/resultados_sonda.zip")